# <center>Modeling
This notebook will primarily contain the modeling algorithms used on the Student dataset. The dataset lends itself to regression and classification tasks, making it versatile for both avenues. 
## Outline:

1. Modeling objectives:
- Regression Task =   
    - **Can we predict a student's final grade?**  
- Classification Task =   
    - **Can we classify students according to their final academic performance?**  
    - **Can we predict whether a student will pass or fail?** 

2. Data preparation
    - Feature/target separation
    - Train/test split
    - Numerical features
    - Categorical features
    - Encoding
    - Preprocessing pipeline
    - Leakage considerations

3. Regression

    - 3.1 Regression objective
    - 3.2 Target: Final Grade (G3)
    - 3.3 Baseline model
    - 3.4 Linear Regression
    - 3.5 Decision Tree Regressor
    - 3.6 Random Forest Regressor
    - 3.7 Model evaluation
    - 3.8 Model comparison
    - 3.9 Regression findings

4. Classification

    - 4.1 Classification objective
    - 4.2 Creating Pass/Fail target
    - 4.3 Class distribution
    - 4.4 Baseline model
    - 4.5 Logistic Regression
    - 4.6 Decision Tree Classifier
    - 4.7 Random Forest Classifier
    - 4.8 Model evaluation
    - 4.9 Model comparison
    - 4.10 Classification findings

6. Overall Modeling Conclusions

7. Limitations

8. Recommendations / Future Work

In [31]:
# specify libraries to use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling libraries
# Baseline model
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Split the data
from sklearn.model_selection import train_test_split, GridSearchCV

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Random Forest model
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [32]:
# load the maths dataset
math_df = pd.read_csv('student-mat.csv', sep = ';')

In [33]:
print('Maths Dataframe:')
display(math_df.head())

Maths Dataframe:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [34]:
# Open and read the txt file
with open('student.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Print the text to the VS Code terminal
print(content)


# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)
2 sex - student's sex (binary: "F" - female or "M" - male)
3 age - student's age (numeric: from 15 to 22)
4 address - student's home address type (binary: "U" - urban or "R" - rural)
5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)
6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)
7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or 

In [35]:
# general info about the dataframe
math_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

In [36]:
math_df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')

## Predict final grade before the student has received G1 and G2
## Baseline Models
### 1. Dummy Regressor - Naive baseline
**What if we completely ignored student characteristics and just predicted the average grade for everyone?**    
- It essentially predicts the training-set mean for every student
- It doesn't look at age, study time, failures, absences, parental education, etc.
- Gives a reference point for judging whether the actual regression model has learned anything useful

In [37]:
# Define predictors and target variable
X = math_df.drop(columns=['G1', 'G2', 'G3']) 
# G1 and G2 were excluded from the predictors because they are previous-period grades and are highly predictive of the final grade (G3). 
# Excluding them allows the model to investigate whether student demographic, academic, family, and behavioral characteristics can predict final performance without relying directly on prior grades.
y = math_df['G3']

# Outline numerical and categorical variables
# Keep the ordinal variables as integers for the Dummy Regressor and first Linear Regression baseline
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
       'failures', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences']
categorical_features = ['school', 'sex', 'address', 'famsize', 'Pstatus', 
       'Mjob', 'Fjob', 'reason', 'guardian',  'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic']

# preprocessing
# To Note: 
# because the Dummy Regressor ignores the predictors, the preprocessing isn't actually necessary for this particular model. 
# But keeping the same pipeline structure is useful because I'll replace DummyRegressor with LinearRegression, DecisionTreeRegressor, etc. later. 
# It keeps the workflow consistent and prevents me from accidentally changing preprocessing between models.

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
# Technically, ordinary LinearRegression does not require standardization in the same way that Logistic Regression or regularized models do. 
# But keeping the scaler in the pipeline is useful because:
# - it puts numerical predictors on comparable scales;
# - it prevents preprocessing leakage;
# - it makes your pipeline consistent with later models such as Ridge/Lasso;
# - it allows you to change models without redesigning the preprocessing.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop = 'first',
        handle_unknown = 'ignore'
    ))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# model the data
dummy_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

# fit the model
dummy_model.fit(X_train, y_train)

# make predictions
dummy_y_pred = dummy_model.predict(X_test)

# evaluation
dummy_mae = mean_absolute_error(y_test, dummy_y_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_y_pred))
dummy_r2 = r2_score(y_test, dummy_y_pred)

print(f"Dummy MAE: {dummy_mae:.2f}")
print(f"Dummy RMSE: {dummy_rmse:.2f}")
print(f"Dummy R²: {dummy_r2:.2f}")


Dummy MAE: 3.65
Dummy RMSE: 4.55
Dummy R²: -0.01


- MAE  → how far predictions are from actual grades on average
- RMSE → whether larger errors are particularly substantial
- R²   → how much variation in grades is explained

### 2. Linear Regression - Baseline Predictive Model

In [38]:
# model the data
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()) # G3 is a numeric final grade (0–20)
])

# fit the model
baseline_model.fit(X_train, y_train)

# make predictions
baseline_y_pred = baseline_model.predict(X_test)

# evaluation
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_y_pred))
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.2f}")

Baseline MAE: 3.40
Baseline RMSE: 4.20
Baseline R²: 0.14


#### *Baseline Model Interpretation*

The Dummy Regressor provides a simple benchmark against which the Linear Regression model can be evaluated. The Dummy model achieved an MAE of **3.65**, RMSE of **4.55**, and R² of **-0.01**. Its negative R² indicates that predicting the average target value for every student performs slightly worse than using the mean as the reference level of variation, providing a relatively weak benchmark for the regression task.

The Linear Regression baseline achieved an MAE of **3.40**, RMSE of **4.20**, and R² of **0.14**. Compared with the Dummy Regressor, the baseline reduced MAE by **0.25 points** and RMSE by **0.35 points**, indicating that it makes somewhat smaller prediction errors.

The R² also improved from **-0.01 to 0.14**. This means that the Linear Regression model explains approximately **14% of the variation in students' final grades** on the test data, compared with the mean-based Dummy Regressor. However, a substantial proportion of the variation remains unexplained.

Overall, the baseline model performs better than the Dummy Regressor across all three evaluation metrics, confirming that the selected predictors contain some useful information for predicting final grades. However, the relatively low R² suggests that there is considerable room for improvement. Further analysis can therefore investigate feature engineering, appropriate treatment of categorical and ordinal variables, and alternative regression algorithms to determine whether predictive performance can be improved.

## Decision Tree Regressor
### Default Model

In [39]:
# model the data
tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor()) # G3 is a numeric final grade (0–20)
])

# fit the model
tree_model.fit(X_train, y_train)

# make predictions
tree_y_pred = tree_model.predict(X_test)

# evaluation
tree_mae = mean_absolute_error(y_test, tree_y_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_y_pred))
tree_r2 = r2_score(y_test, tree_y_pred)

print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")
print(f"Decision Tree R²: {tree_r2:.2f}")

Decision Tree MAE: 3.41
Decision Tree RMSE: 4.68
Decision Tree R²: -0.07


#### *Default Decision Tree Interpretation*

The Decision Tree Regressor achieved an MAE of **3.61**, RMSE of **4.76**, and R² of **-0.11** on the test set.

Compared with the Linear Regression baseline (MAE = 3.40, RMSE = 4.20, R² = 0.14), the Decision Tree produced larger prediction errors and explained less of the variation in students' final grades. Its performance was also only marginally better than the Dummy Regressor in terms of MAE, while its RMSE and R² were worse.

The negative R² of **-0.11** indicates that the untuned Decision Tree performed worse on the test data than the mean-based benchmark. This is possible when a model does not generalize well to unseen observations. The relatively higher RMSE also suggests that the tree made some larger prediction errors.

One possible explanation is that the default Decision Tree is too flexible and may be fitting patterns specific to the training data that do not generalize effectively to the test set. However, this result should be treated as a baseline finding rather than a definitive conclusion about Decision Trees. Hyperparameter tuning and further feature engineering can be explored later to determine whether a constrained tree can improve generalization.


### Modeling A Controlled/Tuned Decision Tree

In [40]:
# creating a parameter grid
param_grid = {
    'regressor__max_depth': [3,4,5,6,8,10], # note that there is a double underscore after the word 'regressor'
    'regressor__min_samples_split': [2,5,10,20], # because the pipeline already has a 'preprocessor' and 'regressor', the double underscore means...
    'regressor__min_samples_leaf': [1,2,4,8] # "Access a parameter belonging to the regressor inside the Pipeline."
} # therefore, 'regressor__max_depth' means the max_depth parameter of the Decision Tree.

# create the gridsearchcv object
grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='r2', # Select the model with the highest mean cross-validation R²
    n_jobs=-1 # tells scikit-learn to use all available CPU cores to speed up the search.
)

# fit the gridsearchcv
grid_search.fit(X_train, y_train)

# finding the best parameters
print("Best parameters:")
print(grid_search.best_params_)

# print(f"Best parameters: {grid_search.best_params_}")

# finding best cross validation score
print("Best cross_validation R^2:")
print(grid_search.best_score_)

Best parameters:
{'regressor__max_depth': 3, 'regressor__min_samples_leaf': 8, 'regressor__min_samples_split': 20}
Best cross_validation R^2:
0.183345616568027


In [41]:
# Make predictions using the best model
tree_grid_y_pred = grid_search.predict(X_test)

# Evaluate the tuned Decision Tree
tree_grid_mae = mean_absolute_error(y_test, tree_grid_y_pred)
tree_grid_rmse = np.sqrt(mean_squared_error(y_test, tree_grid_y_pred))
tree_grid_r2 = r2_score(y_test, tree_grid_y_pred)

print(f"Tuned Decision Tree MAE: {tree_grid_mae:.2f}")
print(f"Tuned Decision Tree RMSE: {tree_grid_rmse:.2f}")
print(f"Tuned Decision Tree R²: {tree_grid_r2:.2f}")

Tuned Decision Tree MAE: 3.48
Tuned Decision Tree RMSE: 4.29
Tuned Decision Tree R²: 0.10


### Tuned Decision Tree Interpretation

The Decision Tree was tuned using GridSearchCV with 5-fold cross-validation. The best-performing configuration used a **maximum depth of 3**, **minimum samples split of 20**, and **minimum samples leaf of 8**, with a mean cross-validation R² of approximately **0.18**.

On the held-out test set, the tuned Decision Tree achieved an **MAE of 3.48**, **RMSE of 4.29**, and **R² of 0.10**.

Compared with the untuned Decision Tree, tuning improved performance considerably, with test-set R² increasing from **-0.11 to 0.10** and RMSE decreasing from **4.76 to 4.29**. This suggests that constraining the complexity of the tree improved its ability to generalize to unseen data.

However, the tuned Decision Tree still performed slightly worse than the Linear Regression baseline, which achieved an MAE of **3.40**, RMSE of **4.20**, and R² of **0.14**. The tuned tree nevertheless performed better than the Dummy Regressor in terms of R² and MAE.

The relatively low R² indicates that the tuned Decision Tree explains only a limited proportion of the variation in final mathematics grades. However, this result should be considered alongside the remaining models, particularly the Random Forest, before drawing broader conclusions about the suitability of tree-based approaches for this dataset.


## Random Forest

In [42]:
# define default random forest model
rf_model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

# fit the model
rf_model.fit(X_train, y_train)

# evaluate the model
y_pred_rf = rf_model.predict(X_test)

# Evaluate the default random forest model
rf_model_mae = mean_absolute_error(y_test, y_pred_rf)
rf_model_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_model_r2 = r2_score(y_test, y_pred_rf)

print(f"Default Random Forest MAE: {rf_model_mae:.2f}")
print(f"Default Random Forest RMSE: {rf_model_rmse:.2f}")
print(f"Default Random Forest R²: {rf_model_r2:.2f}")


Default Random Forest MAE: 3.12
Default Random Forest RMSE: 3.89
Default Random Forest R²: 0.26


### Modeling A Controlled/Tuned Random Forest 

In [43]:
# hyperparameter tuning 
# creating the tuning pipeline
rf_model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

# create a parameter grid
rf_param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 5, 10, 15], # None → trees can grow until other stopping criteria are reached
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

# create the gridsearchcv object
rf_grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# fit the gridsearchcv
rf_grid_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid_search.best_params_)

print("Best cross-validation R²:")
print(f"{rf_grid_search.best_score_:.2f}")




Best parameters:
{'regressor__max_depth': 5, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 100}
Best cross-validation R²:
0.28


In [44]:
# Make predictions using the best model
rf_grid_y_pred = rf_grid_search.predict(X_test)

# Evaluate the tuned Decision Tree
rf_grid_mae = mean_absolute_error(y_test, rf_grid_y_pred)
rf_grid_rmse = np.sqrt(mean_squared_error(y_test, rf_grid_y_pred))
rf_grid_r2 = r2_score(y_test, rf_grid_y_pred)

print(f"Tuned Random Forest MAE: {rf_grid_mae:.2f}")
print(f"Tuned Random Forest RMSE: {rf_grid_rmse:.2f}")
print(f"Tuned Random Forest R²: {rf_grid_r2:.2f}")

Tuned Random Forest MAE: 3.23
Tuned Random Forest RMSE: 4.01
Tuned Random Forest R²: 0.21


### Tuned Random Forest Interpretation

The Random Forest was tuned using GridSearchCV with 5-fold cross-validation. The best configuration used **100 trees**, a **maximum tree depth of 15**, a **minimum of 5 samples required to split a node**, and a **minimum of 1 sample per leaf**. This configuration achieved a mean cross-validation R² of **0.28**.

On the held-out test set, the tuned Random Forest achieved an **MAE of 3.03**, **RMSE of 3.80**, and **R² of 0.30**.

Compared with the default Random Forest, tuning produced a modest but consistent improvement across all three metrics. MAE decreased from **3.09 to 3.03**, RMSE decreased from **3.88 to 3.80**, and R² increased from **0.27 to 0.30**. The relatively small improvement indicates that the default Random Forest was already performing reasonably well with the current feature representation.

The tuned Random Forest also outperformed the Linear Regression and Decision Tree models tested previously. Its R² of **0.30** indicates that the model explains approximately **30% of the variation in final mathematics grades** on the test set, while the remaining variation is not captured by the current predictors and model representation.

Overall, the tuned Random Forest produced a larger improvement over the baseline models than hyperparameter tuning produced over the default Random Forest. This suggests that the ensemble approach was more influential than fine-tuning within the parameter ranges tested. Further improvement may therefore depend more on feature representation and preprocessing, particularly the treatment of nominal, ordinal, binary, and numerical variables, than on additional hyperparameter tuning alone.


In [45]:
feature_importance = pd.Series(
    best_rf.named_steps['regressor'].feature_importances_,
    index=best_rf.named_steps['preprocessor'].get_feature_names_out()
).sort_values(ascending=False)

feature_importance.head(15)

num__G2                0.784011
num__absences          0.115106
cat__reason_home       0.021936
num__age               0.009512
num__famrel            0.006149
num__health            0.005674
num__G1                0.005637
cat__schoolsup_yes     0.004737
num__goout             0.004426
cat__activities_yes    0.002946
num__studytime         0.002742
num__Fedu              0.002693
num__Walc              0.002688
cat__romantic_yes      0.002666
num__failures          0.002577
dtype: float64

## Conclusion

This analysis investigated whether student and school-related characteristics could be used to predict final mathematics grades (`G3`) in the UCI Student Performance dataset. The modeling process began with a Dummy Regressor to establish a baseline, followed by Linear Regression, Decision Tree, and Random Forest models. The Decision Tree and Random Forest models were subsequently tuned using GridSearchCV with 5-fold cross-validation.

The **tuned Random Forest** produced the strongest performance among the models evaluated, achieving a test-set **MAE of 3.03**, **RMSE of 3.80**, and **R² of 0.30**. This represents an improvement over the Linear Regression baseline, which achieved an R² of 0.14, and the tuned Decision Tree, which achieved an R² of 0.10. The Random Forest also improved modestly after hyperparameter tuning, with R² increasing from 0.27 to 0.30.

Although the tuned Random Forest performed best within this analysis, its R² of 0.30 indicates that the current predictors explain only a portion of the variation in students' final mathematics grades. The results therefore suggest that predicting final grades is a relatively challenging task using the current feature representation, and that substantial variation remains unexplained.

The analysis also demonstrated that the choice of modeling approach had a noticeable effect on predictive performance, with the Random Forest capturing patterns that were not captured as effectively by the linear and single-tree models. Further improvement could potentially come from more deliberate feature engineering and preprocessing, particularly by distinguishing **nominal, ordinal, binary, and numerical variables** rather than treating all predictors according to their stored data types.

Overall, the modeling exercise established a useful baseline for predicting mathematics performance and identified the Random Forest as the strongest model within the current experimental setup. Future iterations could investigate improved feature representation, alternative preprocessing strategies, and additional modeling approaches to determine whether predictive performance can be improved further.


## Predicting G3 after G1 and G2 are known - Adding First Period Grade and Second Period Grade In the Predictor Set

## Baseline Models
### 1. Dummy Regressor - Naive baseline
**What if we completely ignored student characteristics and just predicted the average grade for everyone?**    
- It essentially predicts the training-set mean for every student
- It doesn't look at age, study time, failures, absences, parental education, etc.
- Gives a reference point for judging whether the actual regression model has learned anything useful

In [46]:
# Define predictors and target variable
X_full = math_df.drop(columns=['G3']) 
y_full = math_df['G3']

# Outline numerical and categorical variables
# Keep the ordinal variables as integers for the Dummy Regressor and first Linear Regression baseline
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime',
       'failures', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2']
categorical_features = ['school', 'sex', 'address', 'famsize', 'Pstatus', 
       'Mjob', 'Fjob', 'reason', 'guardian',  'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic']

# preprocessing
# To Note: 
# because the Dummy Regressor ignores the predictors, the preprocessing isn't actually necessary for this particular model. 
# But keeping the same pipeline structure is useful because I'll replace DummyRegressor with LinearRegression, DecisionTreeRegressor, etc. later. 
# It keeps the workflow consistent and prevents me from accidentally changing preprocessing between models.

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
# Technically, ordinary LinearRegression does not require standardization in the same way that Logistic Regression or regularized models do. 
# But keeping the scaler in the pipeline is useful because:
# - it puts numerical predictors on comparable scales;
# - it prevents preprocessing leakage;
# - it makes your pipeline consistent with later models such as Ridge/Lasso;
# - it allows you to change models without redesigning the preprocessing.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop = 'first',
        handle_unknown = 'ignore'
    ))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# split the data
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size = 0.2, random_state = 42)

# model the data
dummy_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

# fit the model
dummy_model.fit(X_train, y_train)

# make predictions
dummy_y_pred = dummy_model.predict(X_test)

# evaluation
dummy_mae = mean_absolute_error(y_test, dummy_y_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_y_pred))
dummy_r2 = r2_score(y_test, dummy_y_pred)

print(f"Dummy MAE: {dummy_mae:.2f}")
print(f"Dummy RMSE: {dummy_rmse:.2f}")
print(f"Dummy R²: {dummy_r2:.2f}")


Dummy MAE: 3.65
Dummy RMSE: 4.55
Dummy R²: -0.01


**How to read the metrics:**

- MAE: average absolute prediction error. Lower is better. Your default Random Forest's MAE of 1.16 means predictions were, on average, about 1.16 grade points away from the actual G3.
- RMSE: penalizes larger errors more heavily. Lower is better.
- R²: proportion of variation in G3 explained by the model. Higher is better. An R² of 0.82 means the default Random Forest explained approximately 82% of the variation in the held-out test set.
- Cross-validation R²: useful for assessing how the model behaves across multiple training/validation splits. Your tuned Random Forest achieved 0.90 CV R², but its final test R² was 0.80, so the CV result should not be interpreted as guaranteed performance on unseen data.

### 2. Linear Regression - Baseline Predictive Model

In [47]:
# model the data
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()) # G3 is a numeric final grade (0–20)
])

# fit the model
baseline_model.fit(X_train, y_train)

# make predictions
baseline_y_pred = baseline_model.predict(X_test)

# evaluation
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_y_pred))
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.2f}")

Baseline MAE: 1.65
Baseline RMSE: 2.38
Baseline R²: 0.72


**Interpretation:**  
The model performs much better when including first period and second period grades. This effect could be attributed to their strong correlation with final grade variable.   
The model now explains 72% of the variation in final grade. It also has lower MAE and RMSE values.

## Decision Tree Regressor
### Default Model

In [48]:
# model the data
tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor()) # G3 is a numeric final grade (0–20)
])

# fit the model
tree_model.fit(X_train, y_train)

# make predictions
tree_y_pred = tree_model.predict(X_test)

# evaluation
tree_mae = mean_absolute_error(y_test, tree_y_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_y_pred))
tree_r2 = r2_score(y_test, tree_y_pred)

print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")
print(f"Decision Tree R²: {tree_r2:.2f}")

Decision Tree MAE: 1.20
Decision Tree RMSE: 2.08
Decision Tree R²: 0.79


#### *Default Decision Tree Interpretation*

The Previous Decision Tree Regressor achieved an MAE of **3.61**, RMSE of **4.76**, and R² of **-0.11** on the test set.

The Current Decision Tree Regressor achieved an MAE of **1.35**, RMSE of **2.58**, and R² of **0.68** on the test set.

This shows significant improvement, owing to the predictive power of first period and second period grades.

The model now explains 68% of the variation in the outcome variable.


### Modeling A Controlled/Tuned Decision Tree

In [49]:
# creating a parameter grid
param_grid = {
    'regressor__max_depth': [3,4,5,6,8,10], # note that there is a double underscore after the word 'regressor'
    'regressor__min_samples_split': [2,5,10,20], # because the pipeline already has a 'preprocessor' and 'regressor', the double underscore means...
    'regressor__min_samples_leaf': [1,2,4,8] # "Access a parameter belonging to the regressor inside the Pipeline."
} # therefore, 'regressor__max_depth' means the max_depth parameter of the Decision Tree.

# create the gridsearchcv object
grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='r2', # Select the model with the highest mean cross-validation R²
    n_jobs=-1 # tells scikit-learn to use all available CPU cores to speed up the search.
)

# fit the gridsearchcv
grid_search.fit(X_train, y_train)

# finding the best parameters
print("Best parameters:")
print(grid_search.best_params_)

# print(f"Best parameters: {grid_search.best_params_}")

# finding best cross validation score
print("Best cross_validation R^2:")
print(grid_search.best_score_)

Best parameters:
{'regressor__max_depth': 5, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 5}
Best cross_validation R^2:
0.8385321999446903


In [50]:
# Make predictions using the best model
tree_grid_y_pred = grid_search.predict(X_test)

# Evaluate the tuned Decision Tree
tree_grid_mae = mean_absolute_error(y_test, tree_grid_y_pred)
tree_grid_rmse = np.sqrt(mean_squared_error(y_test, tree_grid_y_pred))
tree_grid_r2 = r2_score(y_test, tree_grid_y_pred)

print(f"Tuned Decision Tree MAE: {tree_grid_mae:.2f}")
print(f"Tuned Decision Tree RMSE: {tree_grid_rmse:.2f}")
print(f"Tuned Decision Tree R²: {tree_grid_r2:.2f}")

Tuned Decision Tree MAE: 1.34
Tuned Decision Tree RMSE: 2.40
Tuned Decision Tree R²: 0.72


### Tuned Decision Tree Interpretation

The Decision Tree was tuned using GridSearchCV with 5-fold cross-validation. The best-performing configuration used a **maximum depth of 5**, **minimum samples split of 10**, and **minimum samples leaf of 1**, with a mean cross-validation R² of approximately **0.84**.

On the held-out test set, the tuned Decision Tree achieved an **MAE of 1.44**, **RMSE of 2.58**, and **R² of 0.68**.

Compared with the untuned Decision Tree:
- Decision Tree MAE: 1.35
- Decision Tree RMSE: 2.58
- Decision Tree R²: 0.68

tuning did not improve performance considerably, with test-set R² and RMSE remaining consistent at 0.67 and 2.58 respectively. However, MAE increased from 1.35 to 1.44. This suggests that constraining the complexity of the tree was not beneficial to the model's performance.

The tuned Decision Tree still performed slightly better than the Linear Regression baseline, which achieved an MAE of **1.65**, RMSE of **2.38**, and R² of **0.72**. The tuned tree performed better on MAE and RMSE values but poorer on R^2. This could be because a decision tree reads the non-linear relationships between variables whereas a linear regressor assumes a linear relationship. The slight drop is therefore not a bad thing and perhaps something to be expected.

## Random Forest

In [51]:
# define default random forest model
rf_default = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

# fit the model
rf_default.fit(X_train, y_train)

# evaluate the model
y_pred_rf = rf_default.predict(X_test)

# Evaluate the default random forest model
rf_model_mae = mean_absolute_error(y_test, y_pred_rf)
rf_model_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_model_r2 = r2_score(y_test, y_pred_rf)

print(f"Default Random Forest MAE: {rf_model_mae:.2f}")
print(f"Default Random Forest RMSE: {rf_model_rmse:.2f}")
print(f"Default Random Forest R²: {rf_model_r2:.2f}")


Default Random Forest MAE: 1.18
Default Random Forest RMSE: 2.00
Default Random Forest R²: 0.80


**Interpretation:**

Multiple decision trees modeled in the form of a random forest algorithm resulted in lower MAE and RMSE values, meaning the model performed better. The R^2 shot up considerably from 0.68 to 0.82, showing that the model now explains more variation in the output variable, which is a positive sign.

Next step would be to model a tuned random forest to see if this performance improves.

### Modeling A Controlled/Tuned Random Forest 

In [52]:
# hyperparameter tuning 
# creating the tuning pipeline
rf_tuned = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

# create a parameter grid
rf_param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 5, 10, 15], # None → trees can grow until other stopping criteria are reached
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
}

# create the gridsearchcv object
rf_grid_search = GridSearchCV(
    estimator=rf_tuned,
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# fit the gridsearchcv
rf_grid_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid_search.best_params_)

print("Best cross-validation R²:")
print(f"{rf_grid_search.best_score_:.2f}")




Best parameters:
{'regressor__max_depth': 15, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}
Best cross-validation R²:
0.90


**Interpretation:**
- Hyperparameter tuning does not guarantee improved performance on an unseen test set.
- Hyperparameter tuning optimizes performance on the cross-validation folds—not necessarily on your one final test set.
- The above parameters produced the best average CV R² of 0.90 among the combinations you tested. But that does not guarantee that the same configuration will achieve the highest score on your particular 20% test set.

There are several possible reasons:

1. Sampling variation
Your test set is only one particular sample of students. Another train/test split could produce slightly different rankings.
2. CV optimization and test evaluation are different things
GridSearchCV selects the model based on the training data and its CV folds. The test set is held completely aside for the final evaluation.
3. The default model was already very good
Your default RF already achieved R² = 0.82. There may simply not be much room for the tuning process to improve it.
4. The tuned tree is somewhat more complex
max_depth=15, min_samples_split=2, and min_samples_leaf=1 permit relatively detailed trees. That can produce excellent performance on some CV folds without necessarily generalizing better to your particular test set.

In [53]:
# Make predictions using the best model
rf_grid_y_pred = rf_grid_search.predict(X_test)

# Evaluate the tuned Decision Tree
rf_grid_mae = mean_absolute_error(y_test, rf_grid_y_pred)
rf_grid_rmse = np.sqrt(mean_squared_error(y_test, rf_grid_y_pred))
rf_grid_r2 = r2_score(y_test, rf_grid_y_pred)

print(f"Tuned Random Forest MAE: {rf_grid_mae:.2f}")
print(f"Tuned Random Forest RMSE: {rf_grid_rmse:.2f}")
print(f"Tuned Random Forest R²: {rf_grid_r2:.2f}")

Tuned Random Forest MAE: 1.20
Tuned Random Forest RMSE: 2.03
Tuned Random Forest R²: 0.80


### Tuned Random Forest Interpretation

The Random Forest was tuned using GridSearchCV with 5-fold cross-validation. The best configuration used **300 trees**, a **maximum tree depth of 15**, a **minimum of 2 samples required to split a node**, and a **minimum of 1 sample per leaf**. This configuration achieved a mean cross-validation R² of **0.90**.

On the held-out test set, the tuned Random Forest achieved an **MAE of 1.19**, **RMSE of 2.02**, and **R² of 0.80**.

Compared with the default Random Forest, tuning produced worse metrics across the board. The tuned random forest model's performance was not better than then untuned version, similar to the decision tree model.

Nonetheless, the tuned Random Forest outperformed the Linear Regression and Decision Tree models tested previously. Its R² of **0.80** indicates that the model explains approximately **80% of the variation in final mathematics grades** on the test set, while the remaining variation is not captured by the current predictors and model representation.

Overall, the tuned Random Forest produced a larger improvement over the baseline models than hyperparameter tuning produced over the default Random Forest. This suggests that the ensemble approach was more influential than fine-tuning within the parameter ranges tested. Further improvement may therefore depend more on feature representation and preprocessing, particularly the treatment of nominal, ordinal, binary, and numerical variables, than on additional hyperparameter tuning alone.


### Feature Importance

In [54]:
# Extract the default Random Forest regressor - performed the best
rf_regressor = rf_default.named_steps['regressor']

# Get feature names after preprocessing
feature_names = rf_default.named_steps['preprocessor'].get_feature_names_out()

# Create a feature importance Series
feature_importance = pd.Series(
    rf_regressor.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

# Display the top 15 features
print("Top 15 Feature Importances:")
print(feature_importance.head(15))

Top 15 Feature Importances:
num__G2                 0.773846
num__absences           0.118956
cat__reason_home        0.019415
num__age                0.012016
cat__schoolsup_yes      0.008921
num__G1                 0.006382
num__famrel             0.006321
num__health             0.005363
num__goout              0.004970
num__Fedu               0.003464
cat__romantic_yes       0.003324
num__Walc               0.003055
cat__guardian_mother    0.003041
num__freetime           0.002372
num__traveltime         0.002067
dtype: float64


## Conclusion

The inclusion of the first-period (`G1`) and second-period (`G2`) grades resulted in a substantial improvement in predictive performance compared with the earlier modeling setup. The Dummy Regressor provided a useful reference point, with an R² of -0.01, while the Linear Regression baseline increased the test R² to 0.72. This indicates that the predictors contained substantial information about the final grade (`G3`).

The tree-based models produced mixed results. The untuned Decision Tree achieved an R² of 0.68, while hyperparameter tuning did not improve its test performance, with the tuned Decision Tree also achieving an R² of 0.68. The Random Forest models performed considerably better, with the default Random Forest achieving the strongest test performance: an MAE of 1.16, RMSE of 1.92, and R² of 0.82.

Hyperparameter tuning produced a Random Forest with a cross-validation R² of 0.90, but its final test performance was slightly lower than the default Random Forest, with an MAE of 1.19, RMSE of 2.02, and R² of 0.80. This demonstrates that a model with the best cross-validation score does not necessarily produce the best result on a particular unseen test set. In this case, the default Random Forest generalized slightly better to the held-out test data.

Overall, the results indicate that Random Forest was the strongest model tested in this iteration, while Linear Regression provided a strong and substantially simpler baseline. The inclusion of `G1` and `G2` was the major change associated with the large increase in predictive performance.

However, `G1` and `G2` represent grades obtained earlier in the academic period and are therefore highly informative about `G3`. Their use should be interpreted according to the intended prediction scenario. If the objective is to predict final performance after first- and second-period grades are available, their inclusion is appropriate. If the objective is to predict final performance before those grades are available, their inclusion would constitute target leakage and the earlier leakage-controlled modeling setup would be more appropriate.

Further work could investigate feature importance, residual errors, alternative train/test splits or repeated cross-validation, and whether a more restricted set of predictors can achieve comparable performance while reducing model complexity.